In [ ]:
import os
import sys

# to setup import paths add project root dirs to sys.path
sys.path.append(os.path.join(os.getcwd(), "..", ".."))
from baseVR.base_functionality import init_import_paths
init_import_paths()


In [ ]:
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# %matplotlib qt
# %matplotlib widget

from analytics_processing import analytics
import analytics_processing.analytics_constants as C
from CustomLogger import CustomLogger as Logger

from dashsrc.plot_components.plot_wrappers.data_selection import group_filter_data

from analytics_processing.modality_loading import session_modality_from_nas
from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args
from analytics_processing.sessions_from_nas_parsing import fullfnames2snames


In [ ]:
%matplotlib widget

In [ ]:
output_dir = "/mnt/SpatialSequenceLearning/Simon/hmm_actions"
data = {}
nas_dir = C.device_paths()[0]
Logger().init_logger(None, None, logging_level="INFO")


In [ ]:
animal_ids = [6]
paradigm_ids = [1100]
session_ids = None

In [ ]:
session_dirs = sessionlist_fullfnames_from_args(paradigm_ids, animal_ids, session_ids)[0]
session_names = fullfnames2snames(session_dirs)

In [ ]:
# framew_beh = analytics.get_analytics('BehaviorFramewise', session_names=session_names[:8],)
framew_beh = analytics.get_analytics('BehaviorFramewise', session_names=session_names,)
framew_beh.columns

In [ ]:
# s_id = '2024-12-11_17-42' #17
# trial_id = 34
# # time_col = 'frame_pc_timestamp'
# time_col = 'frame_ephys_timestamp'

# session_data = framew_beh.xs(s_id, level='session_id')
# trial_data = session_data.loc[session_data['trial_id'] == trial_id]

# trial_data = trial_data.set_index(time_col, drop=True, append=False)
# trial_data.index = (trial_data.index - trial_data.index[0]) /1e6
# trial_data

In [ ]:

def plot_one_trial(trial_data, trial_actions, s_id, trial_id, cluster_labels=None, n_clusters=None):
    """
    Plot a single trial with behavioral metrics and events.
    
    Parameters:
    -----------
    trial_data : pd.DataFrame
        Trial data with time index
    s_id : str
        Session ID
    trial_id : int
        Trial ID
    cluster_labels : array-like, optional
        GMM cluster assignments for each frame (same length as trial_data)
    n_clusters : int, optional
        Number of clusters (required if cluster_labels provided)
    """
    
    # Create figure with optional cluster subplot
    if cluster_labels is not None:
        fig, (ax_clusters, ax_main) = plt.subplots(
            2, 1, figsize=(0.035*len(trial_data), 10), 
            height_ratios=[1, 4],
            sharex=True
        )
        plt.subplots_adjust(hspace=0.05)
    else:
        fig, ax_main = plt.subplots(figsize=(0.035*len(trial_data), 10))

    # main_metric = 'frame_velocity'
    # main_metric = 'frame_raw'
    # main_metric = 'frame_RawYawPitch_abs_vel_sum'

    # which cue was shown
    color = 'orange' if trial_data['cue'].iloc[0] == 1 else 'purple'

    # # cue limits
    # cue_entry_time = trial_data[(trial_data['frame_position'] >-80) & 
    #                             (trial_data['frame_position'] <25)].index.values
    # ax_main.axvspan(cue_entry_time[0], cue_entry_time[-1], color=color, alpha=0.1)
    # # cue limits
    # cue_entry_time = trial_data[(trial_data['track_zone'].isin(['visibleCue', 'nextToCue']))].index.values
    # ax_main.axvspan(cue_entry_time[0], cue_entry_time[-1], color=color, alpha=0.1)

    # R1 limits
    R1_entry_time = trial_data[trial_data['track_zone']=='reward1Zone'].index.values
    ax_main.axvspan(R1_entry_time[0], R1_entry_time[-1], color='grey', alpha=0.2)
    
    # # R2 limits
    # R2_entry_time = trial_data[trial_data['track_zone']=='reward2Zone'].index.values
    # ax_main.axvspan(R2_entry_time[0], R2_entry_time[-1], color='darkgrey', alpha=0.5)

    # r1_thr = (trial_data['velocity_threshold_at_R1'] * trial_data['fps'])
    # r1_frames = r1_thr[(r1_thr.index>R1_entry_time[0]) & (r1_thr.index<R1_entry_time[-1])].index
    # ax_main.plot(r1_thr, linestyle='--', color='black', alpha=.4, label='R1 thr.')
    
    # r2_thr = (trial_data['velocity_threshold_at_R2'] * trial_data['fps'])
    # r2_frames = r2_thr[(r2_thr.index>R2_entry_time[0]) & (r2_thr.index<R2_entry_time[-1])].index
    # ax_main.plot(r2_thr.loc[r2_frames], linestyle='--', color='black', alpha=.4, label='R2 thr.')

    # events
    event_labels = {
        'reward-sound_count': {'label': 'Sound', 'color': 'green', 'size': 80, 'alpha': 0.7, 'marker': '>'},
        'reward-valve-open_count': {'label': 'Reward', 'color': 'green', 'size': 80, 'alpha': 0.7, 'marker': 'o'},
        'lick_count': {'label': 'Lick', 'color': 'black', 'size': 50, 'alpha': 0.4, 'marker': '|'},
        'reward-removed_count': {'label': 'Reward-removed', 'color': 'red', 'size': 80, 'alpha': 0.7, 'marker': 'x'}
    }
    
    for event in event_labels.keys():
        event_data = trial_data[trial_data[event] >= 1]
        if len(event_data) == 0:
            continue
        
        ax_main.scatter(event_data.index, event_data['frame_RawYawPitch_abs_vel_sum_500msMedian'],
        # ax_main.scatter(event_data.index, event_data['frame_RawYawPitch_abs_vel_sum_500msMedian'],
                       color=event_labels[event]['color'], s=event_labels[event]['size'],
                       marker=event_labels[event]['marker'], label=event_labels[event]['label'],
                       alpha=event_labels[event]['alpha'], zorder=5)

    

    # ball_forward_vel	
    # ball_forward_acc	
    # ball_rotation_vel	
    # ball_forward+rotation_vel	
    # ball_forward+rotation_acc	
    # licking	
    # forward_proportion	
    # forward_vs_rotation_corr	
    # forward_std_500ms	
    # rotation_std_500ms
    # below_r_thr = trial_actions['below_r_thr'][trial_actions['below_r_thr'].astype(bool)]
    # ax_main.scatter(below_r_thr.index, np.ones_like(below_r_thr), zorder=1, color='green', label='below_r_thr', s=20)
    
    
    ax_main.plot(trial_actions['ball_forward_vel'], zorder=1, color='blue', label='ball-forward')
    ax_main.plot(trial_actions['ball_forward_acc'], zorder=1, color='blue', alpha=0.4, linestyle=':', label='ball-forward-acc')
    
    ax_main.plot(trial_actions['ball_off_rotation_vel'], zorder=1, color='purple', label='ball-rotation')
    ax_main.plot(trial_actions['ball_off_rotation_acc'], zorder=1, color='purple', alpha=0.4, linestyle=':', label='ball-rotation-acc')
    
    ax_main.plot(trial_actions['ball_forward+rotation_vel'], zorder=1, color='darkblue', label='ball-forward+rotation')
    ax_main.plot(trial_actions['ball_forward+rotation_acc'], zorder=1, color='darkblue', alpha=0.6, linestyle=':', label='ball-forward+rotation-acc')
    
    ax_main.plot(trial_actions['forward_proportion']*100, zorder=1, color='orange', alpha=0.7, linestyle='-', label='forward proportion (%)')
    ax_main.plot(trial_actions['forward_vs_rotation_corr']*100, zorder=1, color='green', alpha=0.7, linestyle='-', label='forward vs rotation corr (%)')
    # ax_main.plot(trial_actions['forward_std_500ms']*10, zorder=1, color='cyan', alpha=0.3, linestyle='-', label='forward std (500ms)')
    # ax_main.plot(trial_actions['rotation_std_500ms']*10, zorder=1, color='magenta', alpha=0.3, linestyle='-', label='rotation std (500ms)')

    # # main metrics
    # ax_main.plot(actions[''], zorder=1, color='blue', label='ball-summed')
    # ax_main.plot(trial_data['frame_RawYawPitch_abs_vel_sum_500msMedian'], zorder=1, color='darkblue', alpha=0.5, linestyle='-', label='ball-summed=smoothed')
    # # ax_main.plot(trial_data['frame_pitch'].abs(), zorder=1, color='blue', alpha=0.3, linestyle=':', label='ball-sideways')
    # ax_main.plot(trial_data['frame_YawPitch_abs_vel_500msMedian'].abs(), zorder=1, color='purple', alpha=0.3, linestyle=':', label='ball-rotation-smoothed')
    # ax_main.plot(trial_data['frame_YawPitch_abs_acc_500msMedian'], zorder=1, color='purple', alpha=0.4, linestyle=':', label='ball-rotation-sm-acc')
    
    # ax_main.plot(trial_data['frame_raw_500msMedian'].abs(), zorder=1, color='blue', alpha=0.3, linestyle=':', label='ball-forward-smoothed')
    # ax_main.plot(trial_data['frame_raw_acc_500msMedian'], zorder=1, color='blue', alpha=0.4, linestyle=':', label='ball-forward-sm-acc')
    # # ax_main.plot(trial_data['facecam_pose_nose_neck_body1_angle'].rolling(6).median(), zorder=1, color='darkred', linestyle=':', label='head-angle')

    ax_main.set_ylim(-50,200)
    ax_main.set_xlabel('Time (s)')
    ax_main.set_ylabel('Velocity [cm/s]')
    ax_main.set_title(f'Session {s_id} - Trial {trial_id}')
    ax_main.legend(ncol=2, fontsize=8)
    ax_main.spines['top'].set_visible(False)
    ax_main.spines['right'].set_visible(False)

    # =========================================================================
    # Cluster visualization subplot (if labels provided)
    # =========================================================================
    if cluster_labels is not None:
        times = trial_data.index.values
        cluster_colors = plt.cm.viridis(np.linspace(0, 1, n_clusters))
        
        # Use scatter plot - one point per frame, colored by cluster
        for c in range(-1, n_clusters):
            mask = cluster_labels == c
            if not mask.any():
                continue
            
            if c == -1:
                color = 'lightgray'
                y_val = -0.3
            else:
                color = cluster_colors[c]
                y_val = c
            
            ax_clusters.scatter(
                times[mask], 
                np.full(mask.sum(), y_val),
                c=[color], 
                s=200,
                marker='|',
                linewidths=2,
            )
            ax_clusters.axhline(y=c, color='lightgray', linestyle='--', linewidth=0.5, zorder=0, alpha=.3)
        
        # Format cluster axis
        ax_clusters.set_ylim(-0.5, n_clusters - 0.5)
        ax_clusters.set_yticks(range(n_clusters))
        ax_clusters.set_yticklabels([f'C{c}' for c in range(n_clusters)], fontsize=8)
        ax_clusters.set_ylabel('Cluster', fontsize=9)
        ax_clusters.spines['top'].set_visible(False)
        ax_clusters.spines['right'].set_visible(False)
        ax_clusters.spines['bottom'].set_visible(False)
        ax_clusters.tick_params(bottom=False)
        
        # Add zone shading to cluster plot too
        # ax_clusters.axvspan(cue_entry_time[0], cue_entry_time[-1], color=color, alpha=0.2)
        ax_clusters.axvspan(R1_entry_time[0], R1_entry_time[-1], color='grey', alpha=0.1)
        # ax_clusters.axvspan(R2_entry_time[0], R2_entry_time[-1], color='darkgrey', alpha=0.2)

    plt.tight_layout()
    # plt.show()
    return fig

In [ ]:
action_cols = [
               # forward velocity and acc
               'frame_raw_500msMedian', 
               'frame_raw_abs_acc_500msMedian', 
               # off rotations velocity
               'frame_YawPitch_abs_vel_sum_500msMedian', 
               'frame_YawPitch_abs_acc_sum_500msMedian', 
               # sum for reward threshold and acc
               'frame_RawYawPitch_abs_vel_sum_500msMedian',
               'frame_RawYawPitch_abs_acc_sum_500msMedian',
               # others
               'lick_detected', 
               
               # derived metrics
               'frame_forward_prop',
               'forward_vs_rotation_corr',
               
               # pose
               'head_angle',
               'head_angle_vel',
               'movement_energy_smooth5',
               
               ]

renamer = {
      'frame_raw_500msMedian': 'ball_forward_vel',
      'frame_YawPitch_abs_vel_sum_500msMedian': 'ball_off_rotation_vel',
      'frame_RawYawPitch_abs_vel_sum_500msMedian': 'ball_forward+rotation_vel',
      'frame_raw_abs_acc_500msMedian': 'ball_forward_acc',
      'frame_YawPitch_abs_acc_sum_500msMedian': 'ball_off_rotation_acc',
      'frame_RawYawPitch_abs_acc_sum_500msMedian': 'ball_forward+rotation_acc',
      'lick_detected': 'licking',
      'frame_forward_prop': 'forward_proportion',
      'forward_vs_rotation_corr': 'forward_vs_rotation_corr',
      'head_angle': 'head_angle',
      'head_angle_vel': 'head_angle_vel',
      'movement_energy_smooth5': 'movement_energy',
      
}
# framew_beh['frame_below_velocity_thr'] = framew_beh['frame_below_velocity_thr'].fillna(0)
actions = framew_beh[action_cols].dropna() # # just for NaN dropping (first ~80 frames of a session)
actions = actions.rename(columns=renamer)
# reorder to value order
actions = actions[renamer.values()]

actions

In [ ]:
actions_cp = actions.copy()
actions_cp['track_zone'] = framew_beh.loc[actions.index, 'track_zone']
actions_cp['trial_id'] = framew_beh.loc[actions.index, 'trial_id']
actions_cp.reset_index(inplace=True)
actions_cp

nframes_pre, nframes_post = 60, 120
def around_r1_entries(group):
    # r1_idx = group[group['track_zone'] == 'reward1Zone'].index[0]
    # return np.arange(r1_idx - nframes_pre, r1_idx + nframes_post)
    r1_idx = group[group['track_zone'] == 'reward1Zone'].index
    if r1_idx.empty:
        print(group[['session_id', 'trial_id']].iloc[0], "No reward1Zone frames found, NaN dropping from pose estimation.")
        return np.array([])  # or handle as needed, e.g., return None or raise an error
    return np.arange(r1_idx[0]-60, r1_idx[-1]+30)

r1_entries = actions_cp.groupby(by=['session_id', 'trial_id']).apply(around_r1_entries, )

r1_entries = np.concatenate(r1_entries.values)
print(f"Slicing to {nframes_pre} frames before and {nframes_post} frames after R1 entry gives us {len(r1_entries)}/{len(actions_cp)} frames to analyze around R1 entries.")

actions = actions.iloc[r1_entries]

In [ ]:
plt.close('all')

s_id = 0
s_id = framew_beh.index.unique('session_id')[s_id]
time_col = 'frame_pc_timestamp'

session_data = framew_beh.loc[framew_beh.index.get_level_values('session_id') == s_id]
for trial_id in session_data['trial_id'].dropna().unique().astype(int):
    if trial_id <40:
        continue
    trial_data = session_data.loc[session_data['trial_id'] == trial_id]
    trial_frame_ids = trial_data.index.intersection(actions.index)
    trial_data = trial_data.loc[trial_frame_ids]
    
    trial_frame_ids = trial_data.index.values
    trial_data = trial_data.set_index(time_col, drop=False, append=False)
    trial_actions = actions.loc[actions.index.isin(trial_frame_ids)]
    trial_data.index = (trial_data.index - trial_data.index[0]) /1e6
    trial_actions.index = trial_data.index
    
    
    fig = plot_one_trial(trial_data, trial_actions, s_id, trial_id)
    plt.show()
    
    if trial_id >50:
        break  

In [ ]:

fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(6, 6), )#sharex=True)
axes = axes.flatten()

actions_scaled = (actions - actions.mean()) / actions.std()

for i, col in enumerate(actions.columns):
    # standardize
    # val = (actions[col] - actions[col].mean()) / actions[col].std()
    # val = actions_scaled[col]
    val = actions[col]
    axes[i].hist(val, bins=30)
    axes[i].set_title(col, fontsize=6)
    axes[i].set_yscale('log')
plt.tight_layout()
plt.show()

In [ ]:

fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(6, 6), )#sharex=True)
axes = axes.flatten()

actions_scaled = (actions - actions.mean()) / actions.std()

for i, col in enumerate(actions.columns):
    # standardize
    # val = (actions[col] - actions[col].mean()) / actions[col].std()
    val = actions_scaled[col]
    # val = actions[col]
    axes[i].hist(val, bins=30)
    axes[i].set_title(col, fontsize=6)
    # axes[i].set_yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# plot correlation matrix
plt.figure(figsize=(8, 6))
corr = actions.corr()

# # cluster correlation matrix using linked hierarchical clustering
# from scipy.cluster.hierarchy import linkage, leaves_list
# linked = linkage(corr, method='ward')
# leaves = leaves_list(linked)
# corr = corr.iloc[leaves, :].iloc[:, leaves]

im = plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title('Correlation Matrix of Action Features')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.patches import Ellipse


def plot_joint_grid(
    df,
    max_points=2000,
    jitter_std=0.01,
    bins=30,
    figsize_per_dim=2.4,
    scatter_alpha=0.25,
    corr_circle_size=500,
    mean_vec=None,          # new: shape (n_features,)
    cov_mat=None,           # new: shape (n_features, n_features), diagonal assumed
    contour_stds=(1.0, 2.0) # new: draw 1σ and 2σ contours
):
    """
    Plot a joint distribution grid with marginals (top row and left column),
    and optional Gaussian contours from mean/cov on each joint panel.
    """

    df = df.copy()
    cols = df.columns
    n = len(cols)

    # ---- Determine which columns need jitter ----
    needs_jitter = {
        col: df[col].value_counts(dropna=False).shape[0] < 5
        for col in cols
    }

    # ---- Precompute jittered data ----
    df_jittered = df.copy()
    for col, apply_jitter in needs_jitter.items():
        if apply_jitter:
            col_range = df[col].max() - df[col].min()
            scale = jitter_std * col_range if col_range > 0 else jitter_std
            df_jittered[col] = df[col] + np.random.randn(len(df)) * scale

    # ---- Precompute correlation matrix ----
    corr = df.corr(method="pearson")

    # ---- Colormap ----
    corr_cmap = cm.get_cmap("RdYlGn")
    norm = plt.Normalize(vmin=-1, vmax=1)

    # ---- Figure setup ----
    fig, axes = plt.subplots(
        nrows=n,
        ncols=n,
        figsize=(figsize_per_dim * n, figsize_per_dim * n),
    )

    # optional index map for mean/cov lookup
    col_to_idx = {c: i for i, c in enumerate(cols)}

    # ---- Plot ----
    for i, row_col in enumerate(cols):
        for j, col_col in enumerate(cols):
            ax = axes[i, j]

            if i == 0 and j > 0:
                # Top row: marginal distributions
                data = df[col_col].dropna()
                unique_vals = data.nunique()
                hist_bins = min(bins, unique_vals) if unique_vals > 1 else 1

                ax.hist(
                    data,
                    bins=hist_bins,
                    density=True,
                    color="gray",
                    alpha=0.7,
                )
                ax.set_title(col_col, fontsize=10)
                ax.set_yticks([])
                for spine in ax.spines.values():
                    spine.set_visible(False)

            elif j == 0 and i > 0:
                # Left column: marginal distributions
                data = df[row_col].dropna()
                unique_vals = data.nunique()
                hist_bins = min(bins, unique_vals) if unique_vals > 1 else 1

                ax.hist(
                    data,
                    bins=hist_bins,
                    density=True,
                    orientation="horizontal",
                    color="gray",
                    alpha=0.7,
                )
                ax.set_ylabel(row_col, fontsize=10)
                ax.set_xticks([])
                for spine in ax.spines.values():
                    spine.set_visible(False)

            else:
                # Joint scatter plot
                if i > 0 and j > 0:
                    sub = df_jittered[[col_col, row_col]].dropna()
                    if len(sub) > max_points:
                        sub = sub.sample(max_points, random_state=42)

                    ax.scatter(
                        sub[col_col],
                        sub[row_col],
                        s=10,
                        alpha=scatter_alpha,
                        linewidths=0,
                    )

                    # ---- Optional Gaussian contours (diagonal cov -> axis-aligned ellipses) ----
                    if mean_vec is not None and cov_mat is not None:
                        ix = col_to_idx[col_col]
                        iy = col_to_idx[row_col]
                        mx = mean_vec[ix]
                        my = mean_vec[iy]
                        sx = np.sqrt(max(cov_mat[ix, ix], 1e-12))
                        sy = np.sqrt(max(cov_mat[iy, iy], 1e-12))
                        # draw outer -> inner so overlap gets darker toward center
                        draw_stds = sorted(contour_stds, reverse=True)
                        fill_alphas = np.linspace(0.06, 0.18, len(draw_stds))  # darker toward center

                        for k, (nsig, a_fill) in enumerate(zip(draw_stds, fill_alphas)):
                            e = Ellipse(
                                (mx, my),
                                width=2 * nsig * sx,
                                height=2 * nsig * sy,
                                angle=0.0,  # diagonal covariance => axis-aligned
                                facecolor="black",
                                edgecolor="black",
                                linewidth=1.6 if k == len(draw_stds) - 1 else 1.0,
                                linestyle="-",
                                alpha=a_fill,
                                zorder=6 + k,
                            )
                            ax.add_patch(e)

                        # crisp center marker
                        ax.scatter(mx, my, s=18, c="black", alpha=0.9, zorder=8)

                    # ---- Correlation circle (kept optional/commented in your original style) ----
                    r = corr.loc[row_col, col_col]
                    circle_color = corr_cmap(norm(r)) if not np.isnan(r) else "lightgray"

                else:
                    # Top-left corner (i==0,j==0)
                    ax.axis("off")

            # spines + grid
            [spine.set_visible(False) for spine in ax.spines.values()]
            ax.minorticks_on()
            ax.grid(True, which="major", linestyle="--", alpha=0.3)
            ax.grid(True, which="minor", linestyle=":", alpha=0.15)

            # ---- Axis labels ----
            if i == n - 1:
                ax.set_xlabel(col_col)
            else:
                ax.set_xlabel("")

            if j == 0 and i > 0:
                pass
            elif j == 0:
                ax.set_ylabel("")
            else:
                ax.set_ylabel("")

    plt.tight_layout()
    return fig, axes

In [ ]:
fig, axes = plot_joint_grid(actions_scaled)
plt.show()
# actions_scaled

In [ ]:
from hmmlearn.hmm import GaussianHMM
from sklearn.model_selection import KFold
import random

# Set global random seed for reproducibility
RANDOM_SEED = 42

def _set_all_seeds(seed):
    """Set all random seeds for full reproducibility."""
    np.random.seed(seed)
    random.seed(seed)
    # Note: hmmlearn uses numpy internally

_set_all_seeds(RANDOM_SEED)

def fit_hmm_models(X, lengths=None, n_components_range=range(2, 15), covariance_type='full', sample_n_trials=1_000,
                   n_iter=200, n_init=3, n_cv_folds=2, random_state=RANDOM_SEED, verbose=False):
    """
    Fit HMM models for a range of component numbers.
    
    Returns all fitted models and selection metrics. Model selection (best_idx) 
    is done separately in plotting/analysis functions.
    
    Parameters:
    -----------
    X : array-like, shape (n_samples, n_features)
        Training data
    lengths : array-like or None
        Lengths of sequences for HMM fitting
    n_components_range : range
        Range of component numbers to try
    covariance_type : str
        'full', 'diag', 'spherical', 'tied'
    n_iter : int
        Max EM iterations per initialization
    n_init : int
        Number of random initializations to try. The best model (highest LL) is kept.
        This helps avoid local optima. Default=3.
    n_cv_folds : int
        Number of cross-validation folds
    random_state : int
        Random seed for reproducibility
    verbose : bool
        Whether to print detailed fitting progress (default=False)
        
    Returns:
    --------
    fitting_results : dict with models, metrics, and metadata
    """
    # CRITICAL: Reset ALL seeds at the start for full reproducibility
    _set_all_seeds(random_state)
    
    print(f"Fitting HMM models on data (shape: {X.shape})")
    print(f"  Covariance type: {covariance_type}, Max iterations: {n_iter}, N initializations: {n_init}")
    print(f"  CV folds: {n_cv_folds}, Random state: {random_state}")
    
    log_likelihoods, log_likelihoods_std, aics, bics, cv_scores, cv_stds, models = [], [], [], [], [], [], []
    n_samples, n_features = X.shape
    kfold = KFold(n_splits=n_cv_folds, shuffle=False)  # shuffle=False for reproducibility with sequences
    
    for n in n_components_range:
        print(f"  n={n:2d}: ", end='')
        
        # Try multiple initializations and keep the best one
        best_model = None
        best_ll = -np.inf
        all_lls = []
        for init_idx in range(n_init):
            
            # first sample lengths (each element is one trial)
            iloc_trials = sorted(np.random.choice(len(lengths), size=min(sample_n_trials, len(lengths)), replace=False)) if lengths is not None else None
            lengths_sample = lengths[iloc_trials] if lengths is not None else None 
            
            # And in the numpy operation:
            # then construct i loc indices for the sampled trials (each element is one frame)
            indices = (np.concatenate([np.arange(start, start + lengths[i]) 
                                    for i, start in enumerate(np.cumsum([0] + lengths.tolist()[:-1]) 
                                    if lengths is not None else [0]) 
                                    if i in iloc_trials]))

            assert np.all(np.diff(indices) >= 0), "Indices should be sorted"

            X_sample = X[indices] if lengths is not None else X
            if verbose:
                print(f"Sampling {len(X_sample):<5d}/ {len(X):<5d} ({100*len(X_sample)/len(X):.1f}%) data points for initialization {init_idx+1}/{n_init}... " )
            
            # Reset seed for this initialization
            _set_all_seeds(random_state + init_idx * 100)
            # _set_all_seeds(random_state)
            
            model = GaussianHMM(n_components=n, covariance_type=covariance_type, 
                               n_iter=n_iter, random_state=random_state + init_idx * 100, 
                               verbose=False)
            try:
                model.fit(X_sample, lengths=lengths_sample) if lengths_sample is not None else model.fit(X_sample)
            except Exception as e:
                print(f"    Initialization {init_idx+1} failed with error: {e}")
                continue
            
            if len(X_sample) == len(X): # no sampling, so we can just use X for scoring
                X_score_on = X_sample
            # else use unseen data for scoring to avoid overfitting bias in LL estimates
            elif lengths is not None:
                # Score on the full data (not just the sampled subset)
                if verbose:
                    print(f"    Scoring on full data for LL evaluation... ", end='')
                X_score_on = X
            
            try:
                ll = model.score(X_score_on)
            except Exception as e:
                print(f"    Scoring failed for initialization {init_idx+1} with error: {e}")
                continue
            if verbose:
                print(f"Fit model with LL: {ll:,.1f}\n")
            
            all_lls.append(ll)
            if ll > best_ll:
                best_ll = ll
                best_model = model
                if verbose:
                    print(f"    Init {init_idx+1}/{n_init}: LL={ll:,.0f} (new best)")
            elif verbose:
                print(f"    Init {init_idx+1}/{n_init}: LL={ll:,.0f}")
        
        models.append(best_model)
        log_likelihoods.append(best_ll)
        log_likelihoods_std.append(np.std(all_lls))
        print(f" Best LL for n={n}: {best_ll:,.0f} (std across inits: {log_likelihoods_std[-1]:,.0f})")
        
        # Compute parameters for AIC/BIC
        if covariance_type == 'full':
            n_cov_params = n * n_features * (n_features + 1) // 2
        elif covariance_type == 'diag':
            n_cov_params = n * n_features
        elif covariance_type == 'spherical':
            n_cov_params = n
        else:
            n_cov_params = n_features * (n_features + 1) // 2
        
        n_params = n*(n - 1) + (n - 1) + n*n_features + n_cov_params
        
        aic = -2 * best_ll + 2 * n_params
        bic = -2 * best_ll + n_params * np.log(n_samples)
        aics.append(aic)
        bics.append(bic)
        print(f"AIC: {aic:,.0f}, BIC: {bic:,.0f}, n_params: {n_params}")
        
        # # Cross-validation - reset seed before CV for reproducibility
        # _set_all_seeds(random_state + 1000)  # Different seed for CV to avoid correlation
        # fold_scores = []
        # for train_idx, test_idx in kfold.split(X):
        #     model_cv = GaussianHMM(n_components=n, covariance_type=covariance_type, 
        #                            n_iter=n_iter, random_state=random_state, verbose=False)
        #     model_cv.fit(X[train_idx])
        #     fold_scores.append(model_cv.score(X[test_idx]))
        # cv_scores.append(np.mean(fold_scores))
        # cv_stds.append(np.std(fold_scores))
        
        # print(f"LL={best_ll:,.0f}, AIC={aic:,.0f}, BIC={bic:,.0f}, CV-LL={cv_scores[-1]:,.0f} (±{cv_stds[-1]:,.0f})")
    
    print(f"\n✓ Fitted {len(models)} models (n_components: {list(n_components_range)}, {n_init} inits each)")
    print(f"  Best by BIC: n={list(n_components_range)[np.argmin(bics)]}")
    
    fitting_results = {
        'models': models,
        'n_components_range': list(n_components_range),
        'log_likelihoods': log_likelihoods,
        'log_likelihoods_std': log_likelihoods_std,
        'aics': aics,
        'bics': bics,
        # 'cv_scores': cv_scores,
        # 'cv_stds': cv_stds,
        'covariance_type': covariance_type,
        'n_samples': n_samples,
        'n_features': n_features,
        'random_state': random_state,
    }
    return fitting_results


def select_and_decode_hmm(X, fitting_results, best_idx=None, sort_by_occurrence=False):
    """
    Select a model and decode states.
    
    Parameters:
    -----------
    X : array-like
        Data used for prediction
    fitting_results : dict
        Output from fit_hmm_models
    best_idx : int or None
        Index into n_components_range. If None, uses argmin(BIC).
    sort_by_occurrence : bool, default=False
        If True, reorder states so that state 0 is the most common, state 1 is 
        second most common, etc. If False, keep original HMM state ordering.
        
    Returns:
    --------
    labels : array, state assignments
    results : dict with model info, parameters, counts
    
    Note:
    -----
    If sort_by_occurrence=True, all returned parameters (means, covars, 
    transition_matrix, startprob) are remapped so that state 0 is the most common.
    If sort_by_occurrence=False, original HMM state ordering is preserved.
    """
    if best_idx is None:
        best_idx = np.argmin(fitting_results['bics'])
    
    n_components_range = fitting_results['n_components_range']
    best_n = n_components_range[best_idx]
    best_model = fitting_results['models'][best_idx]
    raw_labels = best_model.predict(X)
    
    if sort_by_occurrence:
        # # Sort states by occurrence: 0 = most common
        # raw_counts = pd.Series(raw_labels).value_counts().sort_values(ascending=False)
        # raw_state_order = raw_counts.index.tolist()  # e.g., [3, 1, 0, 2] means original state 3 is most common
        # old_to_new = {orig: rank for rank, orig in enumerate(raw_state_order)}  # {3: 0, 1: 1, 0: 2, 2: 3}
        # labels = np.array([old_to_new[l] for l in raw_labels])
        
        # # Remap transition matrix: new_trans[new_i, new_j] = old_trans[old_i, old_j]
        # transition_matrix = best_model.transmat_.copy()
        # remapped_trans = np.zeros_like(transition_matrix)
        # for new_i in range(best_n):
        #     for new_j in range(best_n):
        #         old_i = raw_state_order[new_i]
        #         old_j = raw_state_order[new_j]
        #         remapped_trans[new_i, new_j] = transition_matrix[old_i, old_j]
        
        # # Remap means, covariances, and start probabilities
        # remapped_means = best_model.means_[raw_state_order]
        # remapped_covars = best_model.covars_[raw_state_order]
        # remapped_startprob = best_model.startprob_[raw_state_order]
        
        print(f"  State reordering (by occurrence): {old_to_new}")
    else:
        # Keep original ordering
        labels = raw_labels
        raw_state_order = list(range(best_n))  # [0, 1, 2, ...] - identity mapping
        old_to_new = {i: i for i in range(best_n)}
        remapped_trans = best_model.transmat_.copy()
        remapped_means = best_model.means_.copy()
        remapped_covars = best_model.covars_.copy()
        remapped_startprob = best_model.startprob_.copy()
    
    # Compute counts using the labels
    state_counts = pd.Series(labels).value_counts().sort_index()
    
    switches = np.diff(labels) != 0
    switch_rate = switches.sum() / len(switches)
    
    print(f"✓ Selected model: n_components={best_n} (index={best_idx})")
    print(f"  Sort by occurrence: {sort_by_occurrence}")
    print(f"  State sizes: {state_counts.to_dict()}")
    print(f"  State switch rate: {switch_rate:.3f}")
    
    results = {
        # From fitting
        'n_components_range': n_components_range,
        'log_likelihoods': fitting_results['log_likelihoods'],
        'aics': fitting_results['aics'],
        'bics': fitting_results['bics'],
        'log_likelihoods_std': fitting_results['log_likelihoods_std'],
            # 'cv_scores': fitting_results['cv_scores'],
            # 'cv_stds': fitting_results['cv_stds'],
        'covariance_type': fitting_results['covariance_type'],
        'random_state': fitting_results.get('random_state', 42),
        # Selected model
        'best_n': best_n,
        'best_idx': best_idx,
        'sort_by_occurrence': sort_by_occurrence,
        'state_counts': state_counts,
        'transition_matrix': remapped_trans,
        'startprob': remapped_startprob,
        'means': remapped_means,
        'covars': remapped_covars,
        'switch_rate': switch_rate,
        'old_to_new_mapping': old_to_new,
        'raw_state_order': raw_state_order,
    }
    return labels, results


# Curated color palette - visually distinct, aesthetically pleasing
STATE_COLORS = [
    '#E63946',  # Red (vibrant)
    '#457B9D',  # Steel blue
    '#2A9D8F',  # Teal
    '#E9C46A',  # Saffron/gold
    '#9B5DE5',  # Purple
    '#F77F00',  # Orange
    '#06D6A0',  # Mint green
    '#EF476F',  # Pink
    '#118AB2',  # Cerulean
    '#073B4C',  # Dark teal
    '#8338EC',  # Violet
    '#FB5607',  # Bright orange
    '#3A86FF',  # Dodger blue
    '#FFBE0B',  # Amber
    '#FF006E',  # Magenta
]


def get_state_colors(n_states):
    """Get consistent colors for states from curated palette."""
    if n_states <= len(STATE_COLORS):
        return STATE_COLORS[:n_states]
    cmap = plt.cm.get_cmap('Set2')
    return [cmap(i / n_states) for i in range(n_states)]


def style_axis(ax, grid=True, spines=False):
    """Apply consistent styling: remove spines, add grid."""
    if not spines:
        for spine in ax.spines.values():
            spine.set_visible(False)
    if grid:
        ax.grid(True, alpha=0.7, linestyle='-', linewidth=0.5)
        ax.set_axisbelow(True)


def plot_model_parameters(results, feature_names, output_subdir='hmm_plots'):
    """
    Visualize the Gaussian HMM model parameters for a specific model.
    
    A Gaussian HMM with K states has:
    - K emission distributions: N(μ_k, Σ_k) where μ_k is (n_features,) and Σ_k is (n_features, n_features)
    - Transition matrix A: (K, K) where A[i,j] = P(state_j | state_i)
    - Start probabilities π: (K,)
    
    This creates a dense visualization showing:
    - Row 1: Mean vectors for each state (as bar plots)
    - Row 2: Covariance matrices for each state (as heatmaps)
    - Row 3: Transition matrix and start probabilities
    
    Note: All parameters are already remapped so state 0 = most common, etc.
    """
    import os
    
    viz_dir = f"{output_dir}/{output_subdir}/"
    os.makedirs(viz_dir, exist_ok=True)
    
    n_states = results['best_n']
    means = results['means']  # (n_states, n_features) - already remapped
    covars = results['covars']  # (n_states, n_features, n_features) - already remapped
    trans_matrix = results['transition_matrix']  # already remapped
    startprob = results['startprob']  # already remapped
    state_counts = results['state_counts']  # indexed by new state (0, 1, 2, ...)
    state_colors = get_state_colors(n_states)
    n_features = len(feature_names)
    
    # Create figure: 3 rows
    # Row 1: Mean vectors (one subplot per state)
    # Row 2: Covariance matrices (one subplot per state)
    # Row 3: Transition matrix + start probs + model info
    
    fig = plt.figure(figsize=(3 * n_states + 2, 12))
    
    # Use GridSpec for flexible layout
    gs = fig.add_gridspec(3, n_states + 1, height_ratios=[1, 1.2, 1], 
                          width_ratios=[1] * n_states + [0.4], hspace=0.35, wspace=0.3)
    
    # =========================================================================
    # Row 1: Mean vectors μ_k for each state (k is new remapped state index)
    # =========================================================================
    for k in range(n_states):
        ax = fig.add_subplot(gs[0, k])
        
        y_pos = np.arange(n_features)
        ax.barh(y_pos, means[k], color=state_colors[k], edgecolor='black', alpha=0.8)
        ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
        
        ax.set_yticks(y_pos)
        ax.set_yticklabels(feature_names if k == 0 else [], fontsize=8)
        ax.set_xlabel('Mean value', fontsize=8)
        
        count = state_counts.get(k, state_counts.iloc[k] if k < len(state_counts) else 0)
        pct = count / state_counts.sum() * 100
        ax.set_title(f'State {k}\nμ_{k} ({pct:.1f}%)', fontsize=10, fontweight='bold', 
                    color=state_colors[k])
        style_axis(ax)
        ax.invert_yaxis()
    
    # Colorbar placeholder for row 1
    ax_cb1 = fig.add_subplot(gs[0, -1])
    ax_cb1.axis('off')
    ax_cb1.text(0.5, 0.5, 'Mean\nvectors\nμ_k', ha='center', va='center', fontsize=10, 
               transform=ax_cb1.transAxes)
    
    # =========================================================================
    # Row 2: Covariance matrices Σ_k for each state
    # =========================================================================
    # Find global min/max for consistent colorscale
    all_covars = covars.reshape(-1)
    vmax = np.percentile(np.abs(all_covars), 95)
    vmin = -vmax
    
    for k in range(n_states):
        ax = fig.add_subplot(gs[1, k])
        
        cov_k = covars[k]
        im = ax.imshow(cov_k, cmap='coolwarm', aspect='auto', vmin=vmin, vmax=vmax)
        
        ax.set_xticks(range(n_features))
        ax.set_yticks(range(n_features))
        ax.set_xticklabels([f[:3] for f in feature_names], fontsize=7, rotation=45, ha='right')
        ax.set_yticklabels([f[:3] for f in feature_names] if k == 0 else [], fontsize=7)
        
        ax.set_title(f'Σ_{k}', fontsize=10, fontweight='bold')
        
        # Add diagonal values as text (variances)
        for i in range(n_features):
            ax.text(i, i, f'{cov_k[i,i]:.2f}', ha='center', va='center', 
                   fontsize=6, fontweight='bold', color='black')
    
    # Colorbar for covariances
    ax_cb2 = fig.add_subplot(gs[1, -1])
    plt.colorbar(im, cax=ax_cb2, label='Covariance')
    
    # =========================================================================
    # Row 3: Transition matrix + start probs
    # =========================================================================
    # Transition matrix (takes most of the row)
    ax_trans = fig.add_subplot(gs[2, :n_states-1])
    
    im_trans = ax_trans.imshow(trans_matrix, cmap='Blues', vmin=0, vmax=1)
    ax_trans.set_xticks(range(n_states))
    ax_trans.set_yticks(range(n_states))
    ax_trans.set_xticklabels([f'S{i}' for i in range(n_states)])
    ax_trans.set_yticklabels([f'S{i}' for i in range(n_states)])
    ax_trans.set_xlabel('To state')
    ax_trans.set_ylabel('From state')
    ax_trans.set_title('Transition Matrix A[i,j] = P(s_t=j | s_{t-1}=i)', fontsize=10, fontweight='bold')
    
    for i in range(n_states):
        for j in range(n_states):
            color = 'black' if trans_matrix[i, j] < 0.5 else 'white'
            ax_trans.text(j, i, f'{trans_matrix[i, j]:.2f}', ha='center', va='center', 
                         color=color, fontsize=9)
    
    # Start probabilities
    ax_start = fig.add_subplot(gs[2, n_states-1])
    
    bars = ax_start.bar(range(n_states), startprob, color=state_colors, edgecolor='black')
    ax_start.set_xticks(range(n_states))
    ax_start.set_xticklabels([f'S{i}' for i in range(n_states)])
    ax_start.set_ylabel('Probability')
    ax_start.set_title('Start π', fontsize=10, fontweight='bold')
    ax_start.set_ylim(0, 1)
    style_axis(ax_start)
    
    for bar, p in zip(bars, startprob):
        if p > 0.05:
            ax_start.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                         f'{p:.2f}', ha='center', fontsize=8)
    
    # Model info text
    ax_info = fig.add_subplot(gs[2, -1])
    ax_info.axis('off')
    info_text = f"K={n_states}\n{results['covariance_type']}\ncov"
    ax_info.text(0.5, 0.5, info_text, ha='center', va='center', fontsize=10,
                transform=ax_info.transAxes, family='monospace')
    
    plt.suptitle(f'Gaussian HMM Model Parameters (K={n_states} states, {n_features} features)', 
                fontsize=14, fontweight='bold', y=0.98)
    
    plt.savefig(f"{viz_dir}/00_model_parameters.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print model summary
    print(f"\n{'='*80}")
    print(f"Gaussian HMM Model Structure (K={n_states} states)")
    print(f"{'='*80}")
    print(f"Each state k has an emission distribution: X | state=k ~ N(μ_k, Σ_k)")
    print(f"  • μ_k: mean vector of shape ({n_features},)")
    print(f"  • Σ_k: covariance matrix of shape ({n_features}, {n_features})")
    print(f"  • Covariance type: {results['covariance_type']}")
    print(f"\nTransition dynamics:")
    print(f"  • A[i,j] = P(state_t = j | state_{{t-1}} = i)")
    print(f"  • π[k] = P(state_0 = k) (start probability)")
    print(f"  • Note: States are reordered by occurrence (0=most common)")
    print(f"{'='*80}")


def plot_hmm_results(labels, results, feature_names, output_subdir='hmm_plots', max_points=10_000, random_state=RANDOM_SEED):
    """
    Visualize HMM results with consistent styling.
    
    Creates:
    1. Model selection plot (LL/AIC/BIC curves)
    2. Transition matrix heatmap
    3. State feature profiles (z-scored means) with RdYlGn colormap
    4. Per-feature z-score plots (one subplot per feature)
    5. State duration histograms (normal + log scale per state)
    6. State proportions pie chart
    7. Text summary of states
    
    Note: All state indices are already remapped (0=most common, etc.)
    
    IMPORTANT: 
    - Pie chart shows FRAME counts (total frames in each state)
    - Histogram shows BOUT/SEGMENT counts (number of continuous segments)
    """
    import os
    
    # Set seed for reproducibility in any random operations
    _set_all_seeds(random_state)
    
    viz_dir = f"{output_dir}/{output_subdir}/"
    os.makedirs(viz_dir, exist_ok=True)
    
    n_states = results['best_n']
    state_counts = results['state_counts']  # FRAME counts per state
    trans_matrix = results['transition_matrix']
    
    state_colors = get_state_colors(n_states)
    
    # =========================================================================
    # Compute segment/bout statistics FIRST (needed for multiple plots)
    # =========================================================================
    state_changes = np.diff(labels) != 0
    change_indices = np.where(state_changes)[0] + 1
    change_indices = np.concatenate([[0], change_indices, [len(labels)]])
    
    durations, duration_states = [], []
    for i in range(len(change_indices) - 1):
        start, end = change_indices[i], change_indices[i + 1]
        durations.append(end - start)
        duration_states.append(labels[start])
    
    durations = np.array(durations)
    duration_states = np.array(duration_states)
    
    # Compute segment counts per state
    segment_counts = {s: (duration_states == s).sum() for s in range(n_states)}
    total_segments = len(durations)
    total_frames = len(labels)
    
    # =========================================================================
    # 1. Model selection plot (LL/AIC/BIC)
    # =========================================================================
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    n_range = results['n_components_range']
    
    plot_configs = [
        (results['log_likelihoods'], results['log_likelihoods_std'], 'b', 'Log-Likelihood', 'Log-Likelihood (higher is better)'),
        (results['aics'], np.zeros_like(results['aics']), 'g', 'AIC', 'AIC (lower is better)'),
        (results['bics'], np.zeros_like(results['bics']), 'm', 'BIC', 'BIC (lower is better)'),
    ]

    for ax, (data, std, color, ylabel, title) in zip(axes[:3], plot_configs):
        ax.errorbar(n_range, data, yerr=std, fmt=f'{color}-o', capsize=3, capthick=1)
        ax.plot(n_range, data, f'{color}-o')
        ax.set_xlabel('Number of Components')
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(True, alpha=0.3)
        
    # axes[3].errorbar(n_range, results['cv_scores'], yerr=results.get('cv_stds', None), 
    #                  fmt='c-o', capsize=3, capthick=1)
    axes[3].axvline(results['best_n'], color='r', linestyle='--')
    # axes[3].set_xlabel('Number of states')
    # axes[3].set_ylabel('CV Log-Likelihood')
    # axes[3].set_title('Cross-Validation LL (higher is better)')
    # style_axis(axes[3])
    
    plt.tight_layout()
    plt.savefig(f"{viz_dir}/01_model_selection.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # =========================================================================
    # 2. Transition matrix heatmap
    # =========================================================================
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    
    im = ax.imshow(trans_matrix, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(n_states))
    ax.set_yticks(range(n_states))
    ax.set_xlabel('To state')
    ax.set_ylabel('From state')
    ax.set_title('State Transition Probabilities (reordered by occurrence)')
    
    for i in range(n_states):
        for j in range(n_states):
            color = 'black' if trans_matrix[i, j] < 0.5 else 'white'
            ax.text(j, i, f'{trans_matrix[i, j]:.2f}', ha='center', va='center', color=color)
    
    plt.colorbar(im, ax=ax, label='Probability')
    plt.tight_layout()
    plt.savefig(f"{viz_dir}/02_transition_matrix.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # =========================================================================
    # 3. State feature profiles (z-scored means) - RdYlGn colormap
    # =========================================================================
    means = results['means']  # Already remapped by state occurrence
    means_zscore = (means - means.mean(axis=0)) / (means.std(axis=0) + 1e-8)
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    
    im = ax.imshow(means_zscore.T, cmap='RdYlGn', aspect='auto', vmin=-2, vmax=2)
    ax.set_xticks(range(n_states))
    ax.set_yticks(range(len(feature_names)))
    ax.set_yticklabels(feature_names)
    ax.set_xlabel('State (0=most common)')
    ax.set_ylabel('Feature')
    ax.set_title('State Feature Profiles (z-scored means, reordered by occurrence)')
    
    for i in range(len(feature_names)):
        for j in range(n_states):
            val = means_zscore[j, i]
            text_color = 'black' if -1 < val < 1 else 'white'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', 
                   color=text_color, fontsize=9, fontweight='bold')
    
    plt.colorbar(im, ax=ax, label='Z-score')
    plt.tight_layout()
    plt.savefig(f"{viz_dir}/03_feature_profiles.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # =========================================================================
    # 4. Per-feature z-score plots (5 columns)
    # =========================================================================
    n_features = len(feature_names)
    n_cols = 5
    n_rows = int(np.ceil(n_features / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
    axes = np.atleast_2d(axes)
    
    for feat_idx, feat_name in enumerate(feature_names):
        row, col = feat_idx // n_cols, feat_idx % n_cols
        ax = axes[row, col]
        
        zscores = means_zscore[:, feat_idx]
        y_positions = np.arange(n_states)
        
        bars = ax.barh(y_positions, zscores, color=state_colors, edgecolor='black', height=0.7)
        
        ax.set_yticks(y_positions)
        ax.set_yticklabels([f'State {s}' for s in range(n_states)])
        ax.set_xlabel('Z-score')
        ax.set_title(feat_name, fontweight='bold')
        ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
        ax.set_xlim(-3, 3)
        style_axis(ax)
        
        for bar, zscore in zip(bars, zscores):
            x_pos = bar.get_width() + 0.1 if bar.get_width() >= 0 else bar.get_width() - 0.1
            ha = 'left' if bar.get_width() >= 0 else 'right'
            ax.text(x_pos, bar.get_y() + bar.get_height()/2, f'{zscore:.2f}', 
                   va='center', ha=ha, fontsize=8)
    
    for idx in range(n_features, n_rows * n_cols):
        axes[idx // n_cols, idx % n_cols].set_visible(False)
    
    plt.suptitle('Feature Z-scores by State (0=most common)', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f"{viz_dir}/04a_feature_zscores_per_feature.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # =========================================================================
    # 4b. Per-state z-score plots (one subplot per state, showing all features)
    #     Uses red/green coloring based on z-score sign (like RdYlGn)
    # =========================================================================
    n_cols_state = min(n_states, 4)  # Max 4 columns
    n_rows_state = int(np.ceil(n_states / n_cols_state))
    
    fig, axes = plt.subplots(n_rows_state, n_cols_state, figsize=(5 * n_cols_state, 3 * n_rows_state))
    axes = np.atleast_2d(axes)
    if n_rows_state == 1:
        axes = axes.reshape(1, -1)
    
    # RdYlGn-inspired colors: green for positive, red for negative
    def zscore_to_color(z):
        """Map z-score to RdYlGn-like color: negative=red, positive=green."""
        # Normalize to [-2, 2] range, clip extremes
        z_clipped = np.clip(z, -2, 2)
        # Map to [0, 1]: -2 -> 0 (red), 0 -> 0.5 (yellow), 2 -> 1 (green)
        norm_val = (z_clipped + 2) / 4
        cmap = plt.cm.RdYlGn
        return cmap(norm_val)
    
    for state_idx in range(n_states):
        row, col = state_idx // n_cols_state, state_idx % n_cols_state
        ax = axes[row, col]
        
        zscores = means_zscore[state_idx, :]  # All features for this state
        y_positions = np.arange(n_features)
        
        # Color each bar based on its z-score value
        bar_colors = [zscore_to_color(z) for z in zscores]
        
        bars = ax.barh(y_positions, zscores, color=bar_colors, edgecolor='black', height=0.7)
        
        ax.set_yticks(y_positions)
        ax.set_yticklabels(feature_names, fontsize=9)
        ax.set_xlabel('Z-score')
        
        # Get state count for title
        frame_count = state_counts.get(state_idx, state_counts.iloc[state_idx] if state_idx < len(state_counts) else 0)
        pct = frame_count / total_frames * 100
        ax.set_title(f'State {state_idx} ({pct:.1f}%)', fontweight='bold', color=state_colors[state_idx])
        
        ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
        ax.set_xlim(-3, 3)
        ax.invert_yaxis()  # Top-to-bottom reading order
        style_axis(ax)
        
        # Add z-score labels
        for bar, zscore in zip(bars, zscores):
            x_pos = bar.get_width() + 0.1 if bar.get_width() >= 0 else bar.get_width() - 0.1
            ha = 'left' if bar.get_width() >= 0 else 'right'
            ax.text(x_pos, bar.get_y() + bar.get_height()/2, f'{zscore:.2f}', 
                   va='center', ha=ha, fontsize=8)
    
    # Hide unused subplots
    for idx in range(n_states, n_rows_state * n_cols_state):
        axes[idx // n_cols_state, idx % n_cols_state].set_visible(False)
    
    plt.suptitle('Feature Z-scores by State (red=negative, green=positive)', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f"{viz_dir}/04b_feature_zscores_per_state.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # =========================================================================
    # 5. State duration histograms (normal + log scale per state)
    #    NOTE: n here = number of SEGMENTS/BOUTS, not frames!
    # =========================================================================
    fig, axes = plt.subplots(2, n_states, figsize=(3.5 * n_states, 7))
    if n_states == 1:
        axes = axes.reshape(2, 1)
    
    for state in range(n_states):
        state_durations = durations[duration_states == state]
        n_segments = len(state_durations)
        n_frames = state_counts.get(state, state_counts.iloc[state] if state < len(state_counts) else 0)
        
        if n_segments == 0:
            for row in range(2):
                axes[row, state].text(0.5, 0.5, 'No data', ha='center', va='center', 
                                      transform=axes[row, state].transAxes)
            continue
        
        median_dur, mean_dur = np.median(state_durations), np.mean(state_durations)
        
        for row, (yscale, ylabel, show_legend) in enumerate([
            ('linear', 'Count', True), ('log', 'Count (log)', False)
        ]):
            ax = axes[row, state]
            ax.hist(state_durations, bins=30, color=state_colors[state], 
                   edgecolor='black', alpha=0.7)
            ax.axvline(median_dur, color='red', linestyle='--', linewidth=1.5, 
                      label=f'Median: {median_dur:.0f}')
            ax.axvline(mean_dur, color='blue', linestyle=':', linewidth=1.5, 
                      label=f'Mean: {mean_dur:.1f}')
            ax.set_xlabel('Duration (frames)')
            ax.set_ylabel(ylabel)
            ax.set_yscale(yscale)
            title_suffix = '' if row == 0 else ' (log scale)'
            # Clarify: n_bouts = segments, n_frames = total frames
            ax.set_title(f'State {state}\n({n_segments} bouts, {n_frames:,} frames){title_suffix}', 
                        fontweight='bold', fontsize=9)
            style_axis(ax)
            if show_legend:
                ax.legend(fontsize=8)
    
    plt.suptitle(f'State Duration Distributions ({total_segments} total bouts, {total_frames:,} total frames)', 
                fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f"{viz_dir}/05_duration_histogram_by_state.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # =========================================================================
    # 6. State proportions pie chart - FRAME counts (not segment counts)
    # =========================================================================
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    
    proportions = state_counts / state_counts.sum()
    
    # Use actual state indices from state_counts (in case some states are missing)
    actual_states = list(state_counts.index) if hasattr(state_counts, 'index') else list(range(len(state_counts)))
    n_pie_states = len(proportions)
    pie_colors = [state_colors[s] if s < len(state_colors) else state_colors[s % len(state_colors)] for s in actual_states]
    
    # Use actual frame counts from state_counts
    wedges, texts, autotexts = ax.pie(
        proportions.values, 
        labels=[f'State {s}' for s in actual_states],
        colors=pie_colors,
        autopct=lambda pct: f'{pct:.1f}%\n({int(pct/100 * total_frames):,} frames)',
        startangle=90,
        explode=[0.02] * n_pie_states,
        wedgeprops={'edgecolor': 'black', 'linewidth': 1}
    )
    
    for autotext in autotexts:
        autotext.set_fontsize(9)
        autotext.set_fontweight('bold')
    
    ax.set_title(f'State Occupancy - FRAMES (total: {total_frames:,})\n(0=most common)', 
                fontsize=14, fontweight='bold')
    ax.axis('equal')
    
    plt.tight_layout()
    plt.savefig(f"{viz_dir}/06_state_proportions.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # =========================================================================
    # 7. State switch rate donut chart
    # =========================================================================
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    
    prop_switches = results['switch_rate']
    switch_data = [prop_switches, 1 - prop_switches]
    donut_colors = ['#e74c3c', '#2ecc71']  # Red for switch, green for stay
    pie_labels = [f'Switch\n({prop_switches:.1%})', f'Stay\n({1-prop_switches:.1%})']
    
    wedges, texts, autotexts = ax.pie(
        switch_data, 
        colors=donut_colors, 
        labels=pie_labels,
        autopct='', 
        startangle=90,
        wedgeprops=dict(edgecolor='white', linewidth=2)
    )
    
    # Add center circle for donut effect
    centre_circle = plt.Circle((0, 0), 0.5, fc='white')
    ax.add_artist(centre_circle)
    
    # Center text
    ax.text(0, 0, f'Switch Rate\n{prop_switches:.1%}', ha='center', va='center', 
            fontsize=14, fontweight='bold')
    ax.set_title(f'HMM State Transitions\n(Frame-to-Frame)', fontsize=12, fontweight='bold')
    ax.axis('equal')
    
    plt.tight_layout()
    plt.savefig(f"{viz_dir}/07_switch_rate.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # =========================================================================
    # 8. State summary statistics (text)
    # =========================================================================
    state_profiles = pd.DataFrame(means_zscore, columns=feature_names)
    
    print(f"\n{'='*130}")
    print(f"=== HMM State Summary (n_states={n_states}) - States Ordered by Occurrence (0=most common) ===")
    print(f"{'='*130}")
    print(f"{'State':<8} {'Frames':<12} {'%':<8} {'Bouts':<10} {'Avg Dur':<10} {'Top 3 HIGH features':<40} {'Top 3 LOW features':<40}")
    print(f"{'-'*130}")
    
    for s in range(n_states):
        frame_count = state_counts.get(s, state_counts.iloc[s] if s < len(state_counts) else 0)
        pct = frame_count / total_frames * 100
        bout_count = segment_counts[s]
        avg_dur = frame_count / bout_count if bout_count > 0 else 0
        
        z_scores = state_profiles.iloc[s].sort_values()
        top_high = ', '.join([f"{f}(+{z:.1f})" for f, z in z_scores.tail(3)[::-1].items()])
        top_low = ', '.join([f"{f}({z:.1f})" for f, z in z_scores.head(3).items()])
        
        print(f"S{s:<7} {frame_count:<12,} {pct:>5.1f}%   {bout_count:<10,} {avg_dur:<10.1f} {top_high:<40} {top_low:<40}")
        print(f"Transitions to: " + ", ".join([f"S{j}({trans_matrix[s, j]:.2f})" for j in range(n_states)]))
    
    print(f"{'-'*130}")
    print(f"Total: {total_frames:,} frames, {total_segments:,} bouts")
    print(f"Switch rate: {results['switch_rate']:.3f} (fraction of frames with state change)")
    print(f"Random state used: {results.get('random_state', 'N/A')}")
    print(f"{'='*130}")
    
    print(f"\n✓ Saved HMM visualization plots to {viz_dir}")


def plot_hmm_states_on_embedding(embedding, labels, title_prefix="", output_subdir='hmm_plots', max_points=10_000, random_state=RANDOM_SEED):
    """Plot HMM states on a 2D or 3D embedding.
    
    Note: States are already remapped so state 0 = most common, etc.
    """
    import os
    
    # Set seed for reproducible subsampling
    _set_all_seeds(random_state)
    
    viz_dir = f"{output_dir}/{output_subdir}/"
    os.makedirs(viz_dir, exist_ok=True)
    
    n_states = len(np.unique(labels))
    state_colors = get_state_colors(n_states)
    
    if len(labels) > max_points:
        idx = np.sort(np.random.choice(len(labels), max_points, replace=False))
        embedding_sub, labels_sub = embedding[idx], labels[idx]
    else:
        embedding_sub, labels_sub = embedding, labels
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    for state in range(n_states):
        mask = labels_sub == state
        ax.scatter(embedding_sub[mask, 0], embedding_sub[mask, 1], 
                  s=1, alpha=0.5, label=f'State {state}', color=state_colors[state])
    
    ax.set_xlabel('Dimension 1')
    ax.set_ylabel('Dimension 2')
    ax.set_title(f'{title_prefix} - HMM States (n={n_states}, 0=most common)')
    ax.legend(markerscale=5)
    style_axis(ax)
    
    plt.tight_layout()
    safe_title = title_prefix.lower().replace(' ', '_')
    plt.savefig(f"{viz_dir}/07_{safe_title}_states_2d.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    if embedding.shape[1] >= 3:
        fig = plt.figure(figsize=(12, 10))
        ax = fig.add_subplot(111, projection='3d')
        
        for state in range(n_states):
            mask = labels_sub == state
            ax.scatter(embedding_sub[mask, 0], embedding_sub[mask, 1], embedding_sub[mask, 2],
                      s=1, alpha=0.5, label=f'State {state}', color=state_colors[state])
        
        ax.set_xlabel('Dimension 1')
        ax.set_ylabel('Dimension 2')
        ax.set_zlabel('Dimension 3')
        ax.set_title(f'{title_prefix} - HMM States 3D (n={n_states}, 0=most common)')
        ax.legend(markerscale=5)
        
        plt.tight_layout()
        plt.savefig(f"{viz_dir}/07_{safe_title}_states_3d.png", dpi=150, bbox_inches='tight')
        plt.show()
    
    print(f"✓ Saved embedding plots to {viz_dir}")

In [ ]:
# trial_lengths
lengths = framew_beh.loc[actions.index].reset_index().groupby(['session_id', 'trial_id']).size()
lengths

# # Fit all HMM models
fitting_results_diag3 = fit_hmm_models(
    # X_scaled,
    actions_scaled.values, 
    # lengths=None,
    lengths=lengths.values,
    n_components_range=range(3, 14), 
    covariance_type='diag',
    n_iter=150,
    n_init=5,
    verbose=True,
    sample_n_trials=1500,
)


In [ ]:
# # pickle fitting_results_diag3
# with open(f"{output_dir}/hmm_fitting_results_diag3.pkl", "wb") as f:
#     pickle.dump(fitting_results_diag3, f)
# print(f"✓ Saved HMM fitting results to {output_dir}/hmm_fitting_results_diag3.pkl", end='')

In [ ]:
# # now load
# with open(f"{output_dir}/hmm_fitting_results_diag3.pkl", "rb") as f:
#     fitting_results_diag3 = pickle.load(f)
# print(f"✓ Loaded HMM fitting results from {output_dir}/hmm_fitting_results_diag3.pkl", end='')

In [ ]:
# Select best model (by BIC) and decode states sorted by occurrence
hmm_labels, hmm_results = select_and_decode_hmm(actions_scaled.values, 
                                                fitting_results_diag3, best_idx=4)

# Visualize model parameters (means, covariances, transitions)
plot_model_parameters(hmm_results, feature_names=actions.columns.tolist())

# Visualize all results
plot_hmm_results(
    hmm_labels, hmm_results, 
    feature_names=actions.columns.tolist(),
    output_subdir='hmm_plots'
)


In [ ]:

# def plot_joint_grid(
#     df,
#     max_points=2000,
#     jitter_std=0.01,
#     bins=30,
#     figsize_per_dim=2.4,
#     scatter_alpha=0.25,
#     corr_circle_size=500,
#     mean_vec=None,          # new: shape (n_features,)
#     cov_mat=None,           # new: shape (n_features, n_features), diagonal assumed
#     contour_stds=(1.0, 2.0) # new: draw 1σ and 2σ contours
# ):

# hmm_results["means"][0]
# hmm_results["covars"][0]
state_i = 2
fig, axes = plot_joint_grid(actions_scaled, 
                            mean_vec=hmm_results["means"][state_i],
                            cov_mat=hmm_results["covars"][state_i]
)

In [ ]:
# KL divergence between all HMM latent Gaussian states (both directions)

import numpy as np
import pandas as pd

means = np.asarray(hmm_results["means"])      # (12, 9)
covs  = np.asarray(hmm_results["covars"])     # (12, 9, 9)

n_states, d = means.shape
assert covs.shape == (n_states, d, d), f"{means.shape=}, {covs.shape=}"

def kl_mvn(mu_p, S_p, mu_q, S_q, eps=1e-8):
    """KL( N_p || N_q )"""
    I = np.eye(mu_p.shape[0])
    S_p = S_p + eps * I
    S_q = S_q + eps * I

    sign_p, logdet_p = np.linalg.slogdet(S_p)
    sign_q, logdet_q = np.linalg.slogdet(S_q)
    if sign_p <= 0 or sign_q <= 0:
        raise ValueError("Covariance not positive-definite even after regularization.")

    S_q_inv = np.linalg.inv(S_q)
    diff = (mu_q - mu_p).reshape(-1, 1)

    trace_term = np.trace(S_q_inv @ S_p)
    quad_term  = float(diff.T @ S_q_inv @ diff)
    return 0.5 * (logdet_q - logdet_p - d + trace_term + quad_term)

# Directional matrix: KL_dir[i, j] = KL(state_i || state_j)
KL_dir = np.zeros((n_states, n_states), dtype=float)
for i in range(n_states):
    for j in range(n_states):
        if i != j:
            KL_dir[i, j] = kl_mvn(means[i], covs[i], means[j], covs[j])

state_names = [f"S{i}" for i in range(n_states)]
KL_i_to_j = pd.DataFrame(KL_dir, index=state_names, columns=state_names)
KL_j_to_i = KL_i_to_j.T
KL_sym    = 0.5 * (KL_i_to_j + KL_j_to_i)  # optional symmetric summary

print("KL(i||j):")
display(KL_i_to_j.round(3))

print("KL(j||i):")
display(KL_j_to_i.round(3))

print("Symmetric 0.5*(KL(i||j)+KL(j||i)):")
display(KL_sym.round(3))

plt.close('all')
plt.imshow(np.log(KL_sym+ .00001), cmap='viridis')
plt.colorbar(label='KL Divergence')
plt.show()

In [ ]:
hmm_results['means'].shape, hmm_results['covars'].shape, 

9, 6, 10

In [ ]:
hmm_labels
# diag_hmm_labels = pd.Series(hmm_labels, index=actions.index, name='spherical_hmm_state')
hmm_labels = pd.Series(hmm_labels, index=actions.index, name='diag_hmm_state')
n_states = hmm_labels.value_counts().shape[0]
hmm_labels


In [ ]:
plt.close('all')

s_id = 4
s_id = framew_beh.index.unique('session_id')[s_id]
time_col = 'frame_pc_timestamp'

session_data = framew_beh.loc[framew_beh.index.get_level_values('session_id') == s_id]
for trial_id in session_data['trial_id'].dropna().unique().astype(int):
    trial_data = session_data.loc[session_data['trial_id'] == trial_id]
    
    trial_frame_ids = trial_data.index.intersection(actions.index)
    trial_data = trial_data.loc[trial_frame_ids]

    trial_actions = actions.loc[trial_frame_ids]
    trial_data = trial_data.set_index(time_col, drop=False, append=False)
    trial_data.index = (trial_data.index - trial_data.index[0]) /1e6
    trial_actions.index = trial_data.index
    
    
    trial_clust_assign = hmm_labels.reindex(trial_frame_ids)
    fig = plot_one_trial(trial_data, trial_actions, s_id, trial_id, cluster_labels=trial_clust_assign.values, n_clusters=n_states)
    plt.show()
    
    if trial_id == 3:
        break  


In [ ]:
def calc_state_occupancy(labels,):
    """Calculate state occupancy (proportion of time spent in each state)."""
    occ = pd.Series(labels).value_counts(normalize=True).sort_index()
    return occ

def trans_matrix_from_labels(labels, n_states, return_flat=True, imshow=False):
    states = labels.values
    
    # Initialize transition count matrix
    A_counts = np.zeros((n_states, n_states))
    
    # Count transitions from state i to state j
    for t in range(len(states) - 1):
        from_state = int(states[t])
        to_state = int(states[t + 1])
        A_counts[from_state, to_state] += 1
    
    # Normalize each row to get probabilities
    # Row sums = number of times we left that state
    row_sums = A_counts.sum(axis=1, keepdims=True)
    
    # Avoid division by zero for states that never occur
    with np.errstate(divide='ignore', invalid='ignore'):
        A = A_counts / row_sums
    A[np.isnan(A)] = 0  # Handle 0/0 case
    
    A[np.triu_indices_from(A, k=1)] = np.nan

    if imshow:    
        plt.figure()
        plt.imshow(A, cmap='Blues', vmin=0, vmax=.5)
        plt.show()
    
    if return_flat:
        return A[np.tril_indices_from(A, k=-1)].flatten()
    return A

def inital_state_probs_from_labels(labels, n_states):
    states = labels.values
    start_counts = np.zeros(n_states)
    
    # Count how many times each state is the first state in a sequence
    for t in range(len(states) - 1):
        if t == 0 or states[t-1] != states[t]:  # Start of a new segment
            start_state = int(states[t])
            start_counts[start_state] += 1
    
    # Normalize to get probabilities
    start_probs = start_counts / start_counts.sum()
    return start_probs




In [ ]:
hmm_labels

In [ ]:
# Stacked state occupancy per session (each bar sums to 1)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Build a tidy table: one row per frame with session and state
tmp = pd.DataFrame({
    "session_id": hmm_labels.index.get_level_values("session_id"),
    "state": hmm_labels.values.astype(int),
})

# Count states per session, then convert to densities (row-wise sum = 1)
occ = pd.crosstab(tmp["session_id"], tmp["state"])
occ = occ.reindex(columns=range(n_states), fill_value=0)   # keep consistent state order
occ_density = occ.div(occ.sum(axis=1), axis=0).fillna(0)

# Plot
sessions = [f"{i}: {sid}" for i, sid in enumerate(occ_density.index.astype(str))]
state_colors = get_state_colors(n_states)

fig, ax = plt.subplots(figsize=(max(10, len(sessions) * 0.6), 5))
bottom = np.zeros(len(occ_density))

for s in range(n_states):
    vals = occ_density[s].values
    ax.bar(
        sessions,
        vals,
        bottom=bottom,
        color=state_colors[s],
        edgecolor="white",
        linewidth=0.5,
        label=f"S{s}",
    )
    bottom += vals

ax.set_ylim(0, 1)
ax.set_ylabel("Density / Occupancy")
ax.set_xlabel("Session")
ax.set_title("HMM State Density per Session (stacked, sums to 1)")
ax.legend(title="State", ncol=min(n_states, 6), bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
hmm_labels_trials = hmm_labels.to_frame()
hmm_labels_trials['trial_id'] = framew_beh.loc[hmm_labels.index, 'trial_id']
hmm_labels_trials['cue'] = framew_beh.loc[hmm_labels.index, 'cue']
hmm_labels_trials['trial_outcome'] = framew_beh.loc[hmm_labels.index, 'trial_outcome'].values %5 # only r1 rewards
hmm_labels_trials['choice_R1'] = framew_beh.loc[hmm_labels.index, 'choice_R1'].values

hmm_labels_trials.set_index('trial_outcome', append=True, inplace=True)
hmm_labels_trials.set_index('trial_id', append=True, inplace=True)
hmm_labels_trials.set_index('choice_R1', append=True, inplace=True)
hmm_labels_trials.set_index('cue', append=True, inplace=True)
hmm_labels_trials = hmm_labels_trials.iloc[:,0]
# hmm_labels_trials

In [ ]:
hmm_labels_trials_unique = hmm_labels_trials.reset_index(level='trial_outcome').groupby(level=['session_id', 'trial_id']).first()
outc = hmm_labels_trials_unique['trial_outcome'].values %5 # only r1 rewards

hmm_labels_trials_unique = hmm_labels_trials.reset_index(level='choice_R1').groupby(level=['session_id', 'trial_id']).first()
choice_R1 = hmm_labels_trials_unique['choice_R1'].values

hmm_labels_trials_unique = hmm_labels_trials.reset_index(level='cue').groupby(level=['session_id', 'trial_id']).first()
cue = np.clip(hmm_labels_trials_unique['cue'].values, 1, 2)

hmm_labels_trials_unique = hmm_labels_trials.groupby(level=['session_id', 'trial_id']).first()
trial_id = hmm_labels_trials_unique.index.get_level_values('trial_id')
session_id = hmm_labels_trials_unique.index.get_level_values('session_id')


outc.shape, choice_R1.shape, cue.shape, trial_id.shape, session_id.shape

In [ ]:
# get A and occupancy for each trial-session using groupby
trial_occupancy = hmm_labels_trials.groupby(['session_id', 'trial_id']).apply(calc_state_occupancy).unstack(fill_value=0)
trial_transitions = hmm_labels_trials.groupby(['session_id', 'trial_id']).apply(lambda x: trans_matrix_from_labels(x, n_states=n_states, return_flat=True).flatten())
trial_inital_probs = hmm_labels_trials.groupby(['session_id', 'trial_id']).apply(lambda x: inital_state_probs_from_labels(x, n_states=n_states)).apply(pd.Series)
display(trial_inital_probs.shape)
# trial_occupancy
# trial_transitions
# session_idx, trial_idx = trial_occupancy.index.get_level_values('session_id'), trial_occupancy.index.get_level_values('trial_id')

A_features = np.concatenate(trial_transitions).reshape(len(trial_transitions), -1)
occ_features = trial_occupancy.values
# trial_features = np.concatenate([occ_features, A_features, trial_inital_probs.values], axis=1)
trial_features = np.concatenate([occ_features, A_features], axis=1)
# trial_features = occ_features
print(trial_features.shape)

# pca on trial features
from sklearn.decomposition import PCA
pca = PCA(n_components=3)
# standardize features before PCA
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
trial_features_scaled = scaler.fit_transform(trial_features)
trial_features_pca = pca.fit_transform(trial_features_scaled)
# eigenvalues and explained variance
eigenvalues = pca.explained_variance_
explained_variance_ratio = pca.explained_variance_ratio_
# print("Eigenvalues:", eigenvalues)
print("Explained variance ratio:", explained_variance_ratio)
print(trial_features_pca.shape)


In [ ]:
import umap
import plotly.graph_objects as go
from plotly.subplots import make_subplots


scaler = StandardScaler()
trial_features_scaled = scaler.fit_transform(trial_features)

# PCA for comparison
print("\n=== PCA Analysis ===")
pca = PCA(n_components=min(n_states, trial_features.shape[1]))
pca_embedding = pca.fit_transform(trial_features_scaled)

print(f"PCA explained variance ratio (first 3 components): {pca.explained_variance_ratio_[:3]}")
print(f"PCA cumulative variance (first 3 components): {np.cumsum(pca.explained_variance_ratio_[:3])[-1]:.3f}")
print(f"PCA cumulative variance (all components): {np.cumsum(pca.explained_variance_ratio_)}")

# UMAP embedding
print("\n=== UMAP Analysis ===")
print("Fitting UMAP (this may take a few minutes)...")
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=60,
    min_dist=0.1,
    metric='euclidean',
    random_state=41,
    verbose=True
)
umap_embedding = reducer.fit_transform(trial_features_scaled)
print("UMAP fitting complete! Shape ", umap_embedding.shape)

# # Create color mapping by session
# session_colors = trial_features.index.get_level_values('session_id')
# print(f"\nUnique sessions: {session_colors.unique()}")


In [ ]:
# Manual circular cluster assignment with per-cluster radii
import numpy as np
import pandas as pd

cluster_centers = np.array([
    (-0.15, 9.00),   # cluster 1
    (-4.23, 7.70),    # cluster 2
    (-3.50, 14.00),   # cluster 3
    (-6, 13),  # cluster 4
], dtype=float)

# one radius per center (edit these freely)
cluster_radii = np.array([1.5, 2.5, 1.5, 1.5], dtype=float)

# distances: (n_points, n_clusters)
dists = np.linalg.norm(umap_embedding[:, None, :2] - cluster_centers[None, :, :], axis=2)

# inside each circle
inside = dists <= cluster_radii[None, :]

# default cluster 0 (outside all circles)
cluster_ids = np.zeros(len(umap_embedding), dtype=int)

# if a point is in multiple circles, pick smallest normalized distance (d/r)
norm_d = dists / cluster_radii[None, :]
best_cluster = np.argmin(norm_d, axis=1) + 1  # IDs 1..n_clusters

inside_any = inside.any(axis=1)
cluster_ids[inside_any] = best_cluster[inside_any]

cluster_k_id = pd.Series(cluster_ids, name="cluster_k_id")
cluster_k_id, cluster_k_id.value_counts()

In [ ]:
# save one array for further analyis.
# trial_id session_id outc.shape, choice_R1.shape, cue.shape, trial_id.shape, cluster_k_id

final_labels = pd.DataFrame({
    "session_id": session_id,
    "trial_id": trial_id,
    "cluster_k_id": cluster_k_id.values,
    "outcome": outc,
    "choice_R1": choice_R1,
    "cue": cue
})

# saVE
final_labels.to_csv(os.path.join(output_dir, "trial_cluster_labels.csv"), index=False)
print(f"✓ Saved trial cluster labels to {output_dir}/trial_cluster_labels.csv")
final_labels

In [ ]:
from matplotlib.patches import Circle
from matplotlib.colors import ListedColormap, BoundaryNorm
import numpy as np
import matplotlib.pyplot as plt

plt.close("all")

# prepare values
choice_vals = np.asarray(choice_R1).astype(bool).astype(int)
cluster_vals = np.asarray(cluster_k_id).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(15, 7), sharex=True, sharey=True)

# --- Left: choice_R1 ---
cmap_choice = ListedColormap([plt.cm.viridis(0.0), plt.cm.viridis(1.0)])
norm_choice = BoundaryNorm([-0.5, 0.5, 1.5], cmap_choice.N)

sc_choice = axes[0].scatter(
    umap_embedding[:, 0], umap_embedding[:, 1],
    s=8, alpha=0.65, c=choice_vals, cmap=cmap_choice, norm=norm_choice, linewidths=0
)
cb0 = plt.colorbar(sc_choice, ax=axes[0], fraction=0.046, pad=0.04, ticks=[0, 1])
cb0.set_label("Choice R1")
cb0.set_ticklabels(["skip", "stop"])
axes[0].set_title("UMAP colored by choice")

# --- Right: cluster assignment ---
max_cluster = int(np.nanmax(cluster_vals))
base_cluster_colors = {
    0: "#9E9E9E",  # gray
    1: "#E41A1C",  # red
    2: "#377EB8",  # blue
    3: "#4DAF4A",  # green
    4: "#984EA3",  # purple
}
cluster_colors = [base_cluster_colors.get(i, "#FF7F00") for i in range(max_cluster + 1)]
cmap_cluster = ListedColormap(cluster_colors)
cluster_norm = BoundaryNorm(np.arange(-0.5, max_cluster + 1.5, 1), ncolors=cmap_cluster.N)

sc_cluster = axes[1].scatter(
    umap_embedding[:, 0], umap_embedding[:, 1],
    s=8, alpha=0.65, c=cluster_vals, cmap=cmap_cluster, norm=cluster_norm, linewidths=0
)
cb1 = plt.colorbar(sc_cluster, ax=axes[1], fraction=0.046, pad=0.04)
cb1.set_label("Cluster ID")
if len(np.unique(cluster_vals)) <= 20:
    cb1.set_ticks(np.unique(cluster_vals))
axes[1].set_title("UMAP colored by manual clusters")

# --- circles + annotations on both panels ---
for ax in axes:
    for i, ((cx, cy), r) in enumerate(zip(cluster_centers, cluster_radii), start=1):
        circ = Circle(
            (cx, cy), r,
            facecolor="none", edgecolor="black", linestyle="--",
            linewidth=1.3, alpha=0.95, zorder=10
        )
        # ax.add_patch(circ)
        # ax.annotate(
        #     f"C{i}\n({cx:.2f}, {cy:.2f})\nr={r:.2f}",
        #     xy=(cx, cy),
        #     xytext=(cx + 0.35, cy + 0.35),
        #     fontsize=8,
        #     bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="black", alpha=0.8),
        #     zorder=11
        # )

    # style
    ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.35)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_xlabel("UMAP HMM trial embedding dim1")

axes[0].set_ylabel("UMAP HMM trial embedding dim2")

# keep your view window
# axes[0].set_xlim(-2, 11)
# axes[0].set_ylim(-13, 9)

plt.tight_layout()
plt.show()

In [ ]:
# All-sessions plot + per-session plots (6 columns: first row has extra session-colored panel)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize
from matplotlib.patches import Circle

plt.close("all")

# -----------------------------
# Inputs aligned to umap_embedding
# -----------------------------
session_ids = trial_transitions.index.get_level_values("session_id").to_numpy()
cluster_vals = np.asarray(cluster_k_id).astype(int)
trial_outcome_vals = np.asarray(outc)
cue_vals_raw = np.asarray(cue)
choice_vals_raw = np.asarray(choice_R1)
trial_id_vals = np.asarray(trial_id)

n = len(umap_embedding)
for arr_name, arr in [
    ("session_ids", session_ids),
    ("cluster_k_id", cluster_vals),
    ("trial_outcome", trial_outcome_vals),
    ("cue", cue_vals_raw),
    ("choice_R1", choice_vals_raw),
    ("trial_id", trial_id_vals),
]:
    print(arr_name, "unique values:", np.unique(arr))
    assert len(arr) == n, f"{arr_name} length {len(arr)} != umap_embedding length {n}"

# -----------------------------
# Session indexing for colorbar labels 0,1,2,...
# -----------------------------
unique_sessions = pd.Index(session_ids).unique()
session_to_idx = {sid: i for i, sid in enumerate(unique_sessions)}
session_idx_vals = np.array([session_to_idx[sid] for sid in session_ids], dtype=int)

# -----------------------------
# Global axis limits
# -----------------------------
xmin, xmax = np.percentile(umap_embedding[:, 0], [1, 99])
ymin, ymax = np.percentile(umap_embedding[:, 1], [1, 99])

# -----------------------------
# Custom color mappings
# -----------------------------
# trial_outcome: 0 red, 1 dark green, 2..5 light green
trial_outcome_int = np.rint(trial_outcome_vals).astype(int)
trial_outcome_int = np.where((trial_outcome_int >= 0) & (trial_outcome_int <= 5), trial_outcome_int, 2)
cmap_outcome = ListedColormap([
    "#d62728",  # 0 red
    "#006400",  # 1 dark green
    "#90ee90",  # 2 light green
    "#90ee90",  # 3 light green
    "#90ee90",  # 4 light green
    "#90ee90",  # 5 light green
])
norm_outcome = BoundaryNorm(np.arange(-0.5, 6.5, 1), cmap_outcome.N)

# cue: 1 orange, 2 purple
cue_int = np.rint(cue_vals_raw).astype(int)
cmap_cue = ListedColormap(["orange", "purple"])
norm_cue = BoundaryNorm([0.5, 1.5, 2.5], cmap_cue.N)

# choice: viridis min/max endpoints for binary 0/1
choice_int = np.asarray(choice_vals_raw).astype(bool).astype(int)
cmap_choice = ListedColormap([plt.cm.viridis(0.0), plt.cm.viridis(1.0)])
norm_choice = BoundaryNorm([-0.5, 0.5, 1.5], cmap_choice.N)

# trial_id: continuous viridis
trial_id_float = trial_id_vals.astype(float)
norm_trial = Normalize(vmin=np.nanmin(trial_id_float), vmax=100)

# session index colors (viridis), ticks labeled 0..N-1
session_ticks = np.arange(len(unique_sessions))
if len(unique_sessions) > 1:
    norm_session = BoundaryNorm(np.arange(-0.5, len(unique_sessions) + 0.5, 1), ncolors=plt.get_cmap("viridis").N)
else:
    norm_session = BoundaryNorm([-0.5, 0.5], ncolors=plt.get_cmap("viridis").N)

# cluster: fixed salient colors (0 gray, 1-4 salient), fallback orange for >4
max_cluster = int(np.nanmax(cluster_vals))
base_cluster_colors = {
    0: "#9E9E9E",  # gray
    1: "#E41A1C",  # red
    2: "#377EB8",  # blue
    3: "#4DAF4A",  # green
    4: "#984EA3",  # purple
}
cluster_colors = [base_cluster_colors.get(i, "#FF7F00") for i in range(max_cluster + 1)]
cmap_cluster = ListedColormap(cluster_colors)
cluster_norm = BoundaryNorm(np.arange(-0.5, max_cluster + 1.5, 1), ncolors=cmap_cluster.N)

def _add_circles(ax):
    for (cx, cy), r in zip(cluster_centers, cluster_radii):
        circ = Circle(
            (cx, cy), r,
            facecolor="none",
            edgecolor="black",
            linestyle="--",
            linewidth=1.2,
            alpha=0.9,
            zorder=10
        )
        ax.add_patch(circ)

def _style_ax(ax, title):
    ax.set_title(title, fontsize=10)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel("UMAP-1")
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.25)
    for spine in ax.spines.values():
        spine.set_visible(False)

def plot_umap_six(mask, session_label, show_session_panel=False):
    fig, axes = plt.subplots(1, 6, figsize=(31, 5), sharex=True, sharey=True)
    
    s = 18 if not show_session_panel else 14
    draw_cb = False if not show_session_panel else True

    # 1) cluster_k_id
    sc0 = axes[0].scatter(
        umap_embedding[mask, 0], umap_embedding[mask, 1],
        c=cluster_vals[mask], cmap=cmap_cluster, norm=cluster_norm,
        s=s, alpha=0.65, linewidths=0
    )
    if draw_cb:
        cb0 = plt.colorbar(sc0, ax=axes[0], fraction=0.046, pad=0.04)
        cb0.set_label("Cluster ID")
    uniq_c = np.unique(cluster_vals[mask])
    if len(uniq_c) <= 15 and draw_cb:
        cb0.set_ticks(uniq_c)
    _add_circles(axes[0])
    _style_ax(axes[0], f"{session_label} | cluster_k_id")

    # 2) trial_outcome
    sc1 = axes[1].scatter(
        umap_embedding[mask, 0], umap_embedding[mask, 1],
        c=trial_outcome_int[mask], cmap=cmap_outcome, norm=norm_outcome,
        s=s, alpha=0.65, linewidths=0
    )
    if draw_cb:
        cb1 = plt.colorbar(sc1, ax=axes[1], fraction=0.046, pad=0.04, ticks=[0, 1, 2, 3, 4, 5])
        cb1.set_label("Trial Outcome")
        cb1.set_ticklabels(["0", "1", "2", "3", "4", "5"])
    _add_circles(axes[1])
    _style_ax(axes[1], f"{session_label} | trial_outcome")

    # 3) cue
    sc2 = axes[2].scatter(
        umap_embedding[mask, 0], umap_embedding[mask, 1],
        c=cue_int[mask], cmap=cmap_cue, norm=norm_cue,
        s=s, alpha=0.65, linewidths=0
    )
    if draw_cb:
        cb2 = plt.colorbar(sc2, ax=axes[2], fraction=0.046, pad=0.04, ticks=[1, 2])
        cb2.set_label("Cue")
        cb2.set_ticklabels(["1 (orange)", "2 (purple)"])
    _add_circles(axes[2])
    _style_ax(axes[2], f"{session_label} | cue")

    # 4) choice_R1
    sc3 = axes[3].scatter(
        umap_embedding[mask, 0], umap_embedding[mask, 1],
        c=choice_int[mask], cmap=cmap_choice, norm=norm_choice,
        s=s, alpha=0.65, linewidths=0
    )
    if draw_cb:
        cb3 = plt.colorbar(sc3, ax=axes[3], fraction=0.046, pad=0.04, ticks=[0, 1])
        cb3.set_label("Choice R1")
        cb3.set_ticklabels(["skip", "stop"])
    _add_circles(axes[3])
    _style_ax(axes[3], f"{session_label} | choice_R1")

    # 5) trial_id continuous viridis
    sc4 = axes[4].scatter(
        umap_embedding[mask, 0], umap_embedding[mask, 1],
        c=trial_id_float[mask], cmap="viridis", norm=norm_trial,
        s=s, alpha=0.65, linewidths=0
    )
    if draw_cb:
        cb4 = plt.colorbar(sc4, ax=axes[4], fraction=0.046, pad=0.04)
        cb4.set_label("Trial ID (continuous)")
        _add_circles(axes[4])
    _style_ax(axes[4], f"{session_label} | trial_id")

    # 6) session-index panel: only for all-sessions row; otherwise leave empty
    if show_session_panel:
        sc5 = axes[5].scatter(
            umap_embedding[mask, 0], umap_embedding[mask, 1],
            c=session_idx_vals[mask], cmap="viridis", norm=norm_session,
            s=s, alpha=0.65, linewidths=0
        )
        if draw_cb:
            cb5 = plt.colorbar(sc5, ax=axes[5], fraction=0.046, pad=0.04, ticks=session_ticks)
            cb5.set_label("Session index")
            cb5.set_ticklabels([str(i) for i in session_ticks])
        _add_circles(axes[5])
        _style_ax(axes[5], f"{session_label} | session")
    else:
        axes[5].axis("off")
        axes[5].set_title(f"{session_label} | session", fontsize=10)



    # keep your view window
    axes[0].set_xlim(-2, 11)
    axes[0].set_ylim(-13, 9)

    axes[0].set_ylabel("UMAP-2")
    plt.tight_layout()
    plt.show()

# -----------------------------
# A) First row: all sessions (includes session panel)
# -----------------------------
all_mask = np.ones(n, dtype=bool)
plot_umap_six(all_mask, "All sessions", show_session_panel=True)

# -----------------------------
# B) Per-session rows: leave session panel empty
# -----------------------------
for sess_i, sid in enumerate(unique_sessions):
    mask = session_ids == sid
    if mask.sum() == 0:
        continue
    session_label = f"Session {sess_i}: {sid}"
    plot_umap_six(mask, session_label, show_session_panel=False)
    # break

In [ ]:
plt.close("all")
# umap_embedding
# final_labels
# actions_scaled#.groupby(level=['session_id', 'trial_id']).size()

trial_ephys_cols = framew_beh.loc[actions_scaled.index, ['trial_id', 'frame_ephys_timestamp']].reset_index().drop(columns=['paradigm_id', 'animal_id', 'entry_id'])

from_t, delta_t = 400_000, 600_000 # us
def make_interval(group):
    start = group['frame_ephys_timestamp'].iloc[0] + from_t
    end = start + delta_t
    if pd.isna(start) or pd.isna(end):
        return np.nan
    print(pd.Interval(start, end, closed='both'))
    return pd.Interval(start, end, closed='both')
trial_ephys_intervals = trial_ephys_cols.groupby(['session_id', 'trial_id']).apply(make_interval)
trial_ephys_intervals

In [ ]:
fr_z = analytics.get_analytics('FiringRate40msZ', session_names=session_names,)
fr_z = fr_z.reset_index().drop(columns=['paradigm_id', 'animal_id', 'entry_id', 'to_ephys_timestamp'])#.set_index(['from_ephys_timestamp',])
fr_z

In [ ]:
choice_interval_fr_z = []
for s_id in framew_beh.index.unique('session_id'):
    s_fr_z = fr_z[fr_z['session_id'] == s_id].copy()
    if s_fr_z.empty:
        continue
    # turn into pd intervalindex
    idx = pd.IntervalIndex(trial_ephys_intervals[s_id].values)
    
    trial_assignm = pd.cut(
        s_fr_z['from_ephys_timestamp'],
        bins=idx,
        labels=trial_ephys_intervals[s_id].index.get_level_values('trial_id')
    )
    # s_fr_z['trial_id_choice_interval'] = trial_assignm
    s_fr_z['trial_id'] = trial_assignm.cat.codes
    s_fr_z = s_fr_z[s_fr_z['trial_id'] != -1]
    
    s_fr_z = s_fr_z.set_index(['session_id', 'trial_id', 'from_ephys_timestamp'])
    choice_interval_fr_z.append(s_fr_z)

choice_interval_fr_z = pd.concat(choice_interval_fr_z)
choice_interval_fr_z.groupby(level=['session_id', 'trial_id']).size()

In [ ]:
session_wise_pca = []

for s_id in framew_beh.index.unique('session_id'):
    if s_id not in choice_interval_fr_z.index.get_level_values('session_id'):
        continue
    s_fr_z = choice_interval_fr_z.loc[s_id].dropna(axis=1, how='all')
    
    pca = PCA(n_components=10)
    pca_proj = pca.fit_transform(s_fr_z)
    session_wise_pca.append(pca_proj)
    
session_wise_pca = np.concatenate(session_wise_pca, axis=0)
print(session_wise_pca.shape)
choice_interval_pca_proj = pd.DataFrame(session_wise_pca, index=choice_interval_fr_z.index, columns=[f'pca_{i}' for i in range(session_wise_pca.shape[1])])
choice_interval_pca_proj

In [ ]:
ens = analytics.get_analytics('ConcatenatedEnsambleProj40ms', session_names=session_names,)
ens.set_index(['session_id', 'from_ephys_timestamp'], inplace=True, drop=True)
ens = ens.loc[choice_interval_pca_proj.index.droplevel('trial_id')]
ens.index = choice_interval_pca_proj.index
ens.columns = [f'ens_{i}' for i in range(ens.shape[1])]
ens


In [ ]:
plt.close("all")

def plot_session_pca_proj_on_UMAP(session_name, umap_values, choice_labels, pca_proj):
    # normalize inputs
    umap_values = np.asarray(umap_values)
    if umap_values.ndim != 2 or umap_values.shape[1] < 2:
        raise ValueError(f"umap_values must have shape (n_samples, >=2), got {umap_values.shape}")

    choice_vals = np.asarray(choice_labels).astype(bool).astype(int)

    if not isinstance(pca_proj, pd.DataFrame):
        pca_proj = pd.DataFrame(pca_proj)

    n_extra = pca_proj.shape[1]
    n_panels = 1 + n_extra

    fig, axes = plt.subplots(1, n_panels, figsize=(max(6, 2.8 * n_panels), 3), sharex=True, sharey=True)
    if n_panels == 1:
        axes = np.array([axes])

    # --- Left: choice_R1 ---
    cmap_choice = ListedColormap([plt.cm.viridis(0.0), plt.cm.viridis(1.0)])
    norm_choice = BoundaryNorm([-0.5, 0.5, 1.5], cmap_choice.N)

    sc_choice = axes[0].scatter(
        umap_values[:, 0], umap_values[:, 1],
        s=25, alpha=0.85, c=choice_vals, cmap=cmap_choice, norm=norm_choice, linewidths=0
    )
    cb0 = plt.colorbar(sc_choice, ax=axes[0], fraction=0.046, pad=0.04, ticks=[0, 1])
    cb0.set_label("Choice R1")
    cb0.set_ticklabels(["skip", "stop"])
    axes[0].set_title("Choice", fontsize=8)

    # style first panel
    axes[0].grid(True, linestyle="--", linewidth=0.6, alpha=0.35)
    for spine in axes[0].spines.values():
        spine.set_visible(False)
    axes[0].set_xlabel("UMAP HMM trial emb. dim1")

    # same scatter plot but color by any provided columns (variable count / names)
    for panel_idx, col_name in enumerate(pca_proj.columns, start=1):
        col_vals = pca_proj[col_name].values
        vmax = np.nanpercentile(np.abs(col_vals), 98)
        vmax = 2.0 #if (not np.isfinite(vmax) or vmax == 0) else vmax

        sc = axes[panel_idx].scatter(
            umap_values[:, 0], umap_values[:, 1],
            s=25, alpha=0.85, c=col_vals, cmap='RdBu_r', vmin=-vmax, vmax=vmax, linewidths=0
        )

        axes[panel_idx].set_title(str(col_name), fontsize=8)
        axes[panel_idx].grid(True, linestyle="--", linewidth=0.6, alpha=0.35)
        for spine in axes[panel_idx].spines.values():
            spine.set_visible(False)
        axes[panel_idx].set_xlabel("UMAP HMM trial emb. dim1")

        # only one colorbar on last panel to reduce clutter
        if panel_idx == n_panels - 1:
            plt.colorbar(sc, ax=axes[panel_idx], fraction=0.046, pad=0.04)

    axes[0].set_ylabel("UMAP HMM trial emb. dim2")

    plt.suptitle(session_name, fontsize=12)
    plt.tight_layout()
    plt.show()


all_choices, all_umap, all_pca_proj = [], [], []
for sess_i, sid in enumerate(final_labels.session_id.unique()):
    session_name = f"Session {sess_i}: {sid}"
    if sid not in choice_interval_pca_proj.index.unique('session_id'):
        continue

    mask = final_labels.session_id == sid
    sess_choiceR1 = final_labels.loc[mask, 'choice_R1']
    sess_umap = umap_embedding[mask]

    # sess_pca_proj_timemean = choice_interval_pca_proj.loc[sid].groupby(level='trial_id').mean()
    sess_pca_proj_timemean = ens.loc[sid].groupby(level='trial_id').mean()

    # optional filtering; still works with arbitrary column names
    column_mask = sess_pca_proj_timemean.mean().abs().values > .2
    sess_pca_proj_timemean = sess_pca_proj_timemean.loc[:, column_mask]

    print(sess_pca_proj_timemean.shape)

    if sess_umap.shape[0] != sess_pca_proj_timemean.shape[0]:
        print(f"Skipping {session_name} ephys shorter than behavior...")
        continue

    plot_session_pca_proj_on_UMAP(session_name, sess_umap, sess_choiceR1, sess_pca_proj_timemean)
    all_choices.append(sess_choiceR1)
    all_umap.append(sess_umap)
    all_pca_proj.append(sess_pca_proj_timemean)
    # break

# # concatenate all sessions for overall plot
# all_choices = pd.concat(all_choices)
# all_umap = np.concatenate(all_umap, axis=0)
# all_pca_proj = pd.concat(all_pca_proj)
# plot_session_pca_proj_on_UMAP("All sessions combined", all_umap, all_choices, all_pca_proj)